In [1]:
# %pip install \
#     --extra-index-url=https://pypi.nvidia.com \
#     cudf-cu12==24.6.* dask-cudf-cu12==24.6.* cuml-cu12==24.6.* \
#     cugraph-cu12==24.6.* cuspatial-cu12==24.6.* cuproj-cu12==24.6.* \
#     cuxfilter-cu12==24.6.* cucim-cu12==24.6.* pylibraft-cu12==24.6.* \
#     raft-dask-cu12==24.6.* cuvs-cu12==24.6.*

In [2]:
# pip install fastai torchmetrics seaborn xgboost pandas peft

In [ ]:
%load_ext cudf.pandas

# fastai / deeplearning
from fastai.tabular.all import *
from torchmetrics.classification import BinaryMatthewsCorrCoef

# Standard library imports
import gc
import os
import time
from pathlib import Path
from types import SimpleNamespace

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import matthews_corrcoef

# trees
import xgboost as xgb

sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)
np.set_printoptions(linewidth=140)
pd.set_option('display.width', 140)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

competition = 'playground-series-s4e8'
path = Path('/kaggle/input' if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ else '.')/competition

sample_df = pd.read_csv(path/'sample_submission.csv')
test_df = pd.read_csv(path/'test.csv')
original_df = pd.read_csv('one_million_mushrooms.csv', sep=';')
df = pd.read_csv(path/'train.csv')

# DEBUG - Just to test run the pipeline
small_df = df.sample(n=1000)
df.head()

In [ ]:
cont_cols = ["cap-diameter", "stem-height", "stem-width"]
cat_cols = list(set(df.columns) - set(cont_cols) - {"id", "class"})
features = cat_cols + cont_cols

# Config Object

In [ ]:
CFG = SimpleNamespace(
    valid_pct = 0.05,
    optimal_threshold = 0.5, # fine tune this
    bs = 2048,
)

# Using Fastai TabularPandas For Preprocessing

In [ ]:
def clean_artifacts(df, cat_cols, replace_threshold=0.001):
    for col in cat_cols:
        if pd.api.types.is_object_dtype(df[col]):
            value_counts = df[col].value_counts()
            infrequent = value_counts[value_counts < len(df) * replace_threshold].index
            df[col] = df[col].where(~df[col].isin(infrequent), 'rare')
            df[col] = df[col].where(~df[col].str.lower().isin(['none', 'nan', 'na']), np.nan)
    return df

def get_tabular_object(df):
    df = df.copy()
    df = df.fillna(df.mode().iloc[0])
    df = clean_artifacts(df, cat_cols) 
    return TabularPandas(
        df, splits=RandomSplitter(valid_pct=CFG.valid_pct)(df),
        procs = [Categorify, FillMissing, Normalize],
        cat_names=cat_cols,
        cont_names=cont_cols,
        y_names="class", y_block = CategoryBlock(),
    )

def get_dataloader(df):
    return get_tabular_object(df).dataloaders(path=path, bs=CFG.bs)

# to = get_tabular_object(original_df)
# dataloader = to.dataloaders(path=path, bs=CFG.bs)
# dataloader.show_batch(max_n=3)

# Fastai Metric - MatthewsCorrCoeff

In [ ]:
class FastaiBinaryMCC(Metric):
    def __init__(self, threshold, device):
        self.mcc_metric = BinaryMatthewsCorrCoef(threshold=threshold).to(device)
    
    def reset(self):
        self.mcc_metric.reset()
    
    def accumulate(self, learn):
        self.mcc_metric.update(learn.pred[:, 1].unsqueeze(1), learn.y)
    
    @property
    def value(self):
        return self.mcc_metric.compute()
    
    @property
    def name(self):
        return "MCC"

# Embeddings Neural Net

In [ ]:
def get_fastai_learner(final_layers=[128,64,32]):
    mcc_metric = FastaiBinaryMCC(threshold=CFG.optimal_threshold, device=device)
    return tabular_learner(get_dataloader(original_df), metrics=mcc_metric, layers=final_layers).to_fp16()
    
learn = get_fastai_learner()
learn.path = Path('/kaggle/working')
learn.model

That's the rule that define the embedding size

In [ ]:
emb_sz_rule??

In [ ]:
# suggested_lrs = learn.lr_find(suggest_funcs=(slide, valley))
# optimal_lr = (suggested_lrs.slide + suggested_lrs.valley) / 2
# optimal_lr

In [ ]:
# learn.fit_one_cycle(3, lr_max=suggested_lrs.slide)

In [ ]:
# learn.save('pre-trained')
# learn.export('pre-trained.pkl')

# Lora

In [ ]:
from torch.utils.data import Dataset
from torch.utils.data import random_split


def split_tabular_dataset(dataset, val_fraction=0.05):
    total_size = len(dataset)
    val_size = int(total_size * val_fraction)
    train_size = total_size - val_size
    train_dataset, val_dataset = random_split(
        dataset, 
        [train_size, val_size],
        generator=torch.Generator()
    )
    return train_dataset, val_dataset


class TabularDataset(Dataset):
    def __init__(self, tabular_pandas):
        self.cont_data = tabular_pandas.conts.values
        self.cat_data = tabular_pandas.cats.values
        self.targets = tabular_pandas.y.values
        self.cont_cols = tabular_pandas.cont_names
        self.cat_cols = tabular_pandas.cat_names
        
    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        x_cont = torch.tensor(self.cont_data[idx], dtype=torch.float)
        x_cat = torch.tensor(self.cat_data[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx], dtype=torch.long)
        return {"x_cont": x_cont, "x_cat": x_cat, "labels": y}


tabular_ds = TabularDataset(get_tabular_object(df))
train_dataset, val_dataset = split_tabular_dataset(tabular_ds)
print(f"Total dataset size: {len(tabular_ds)}")
print(f"Training set size: {len(train_dataset)}")
print(f"Validation set size: {len(val_dataset)}")

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType
from transformers import Trainer, TrainingArguments
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_features, out_features, r=8, alpha=16):
        super().__init__()
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        
        self.lora_A = nn.Parameter(torch.zeros(in_features, r))
        self.lora_B = nn.Parameter(torch.zeros(r, out_features))
        
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        return x + (x @ self.lora_A @ self.lora_B) * self.scaling

class LoRALinear(nn.Module):
    def __init__(self, linear, r=8, alpha=16):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(linear.in_features, linear.out_features, r, alpha)
    
    def forward(self, x):
        return self.linear(x) + self.lora(x)


def add_lora_to_linear(layer, rank=4, alpha=1):
    if isinstance(layer, nn.Linear):
        in_features, out_features = layer.in_features, layer.out_features
        lora = LoRALayer(in_features, out_features, rank, alpha)
        return nn.Sequential(layer, lora)
    return layer


def add_lora_layers(model, r=8, alpha=16):
    for module in model.modules():
        if isinstance(module, LinBnDrop):
            if isinstance(module[0], nn.Sequential):
                linear = module[0][0]
                lora_linear = LoRALinear(linear, r, alpha)
                module[0] = nn.Sequential(lora_linear)
    return model


def freeze_non_lora_parameters(module):
    for name, param in module.named_parameters():
        if 'lora_A' not in name and 'lora_B' not in name:
            param.requires_grad = False
        else:
            param.requires_grad = True



gc.collect()
torch.cuda.empty_cache()
learn = load_learner('/kaggle/working/pre-trained.pkl')
# Apply LoRA to the model
model_with_lora = add_lora_layers(learn.model)
print("Lora Architecture:")
display(model_with_lora)


# # Freeze non-LoRA parameters
learn.model = model_with_lora
freeze_non_lora_parameters(learn.model)
learn.dls = get_dataloader(df)
learn.fit_one_cycle(5, 1e-3)

In [ ]:
def get_test_dl(test_df):
    test_df = test_df.copy()
    test_df = test_df.fillna(test_df.mode().iloc[0])
    test_df = clean_artifacts(test_df, cat_cols)
    return learn.dls.test_dl(test_df)

test_dl = get_test_dl(test_df)
test_preds, _ = learn.get_preds(dl=test_dl)
print('test_preds', test_preds[:5])
final_preds = (test_preds[:, 1] > CFG.optimal_threshold)
print('final_preds', final_preds[:5])

# Bonus: fastai + trees (optional)

In [ ]:
# def xgboost_model(train_x, train_y, valid_x, valid_y):
#     params = {
#         'objective': 'binary:logistic',
#         'eval_metric': 'logloss',
#         'n_estimators': 300, # fine tune this
#         'learning_rate': 0.01,
#         'max_depth': 6,
#         'reg_alpha': 1,
#         'reg_lambda': 1,
#         'subsample': 0.6,
#         'colsample_bytree': 0.6,        
#         'use_label_encoder': False,
#     }
#     model = xgb.XGBRegressor(**params)
#     model.fit(
#         train_x, train_y,
#         eval_set=[(train_x, train_y), (valid_x, valid_y)],
#         verbose=10
#     )
#     return model


# xgb_model = xgboost_model(to.train.xs, to.train.y, to.valid.xs, to.valid.y)

# valid_preds = xgb_model.predict(to.valid.xs)
# valid_preds_binary = (valid_preds > CFG.optimal_threshold).astype(int)
# final_mcc = matthews_corrcoef(to.valid.y, valid_preds_binary)
# print(f"final MCC: {final_mcc}")

# Ensembling (optional)

In [ ]:
def ensemble(epochs):
    learn = get_fastai_learner()
    learn.fit_one_cycle(epochs, lr_max=suggested_lrs.slide)
    return learn.get_preds(dl=test_dl)[0]


gc.collect()
torch.cuda.empty_cache()
epochs = 16 # <------------------------------------------------- Try with different epochs
all_preds = [ensemble(epochs) for _ in range(5)]
ens_preds = torch.stack(all_preds).mean(0)
final_preds = (ens_preds[:, 1] > CFG.optimal_threshold)
final_preds.shape, final_preds[0]

# Submission

In [ ]:
submission = pd.DataFrame({
    'id': test_df['id'],
    'class': pd.Series(final_preds).map({False: 'e', True: 'p'})
})
model_name = type(learn.model).__name__
submission.to_parquet('submission.parquet', index=False)
submission.to_parquet(f'mushroom_subm_{model_name}.parquet', index=False)
submission.to_csv('submission.csv', index=False)

In [ ]:
plt.pie(submission['class'].value_counts(), autopct='%1.1f%%', labels=['e', 'p'])
plt.title('Distribution of Classes')
plt.show()